# vLLM vs. native `transformers` -- Ouro early-exit comparison

Runs the SAME model/dataset/examples through both `inference_and_eval.inference.vllm_structured_infer.run()` and the new `inference_and_eval.inference.native_infer.run()`, so their completions can be diffed directly and both scored with the unchanged `inference_and_eval.eval.run_eval`.

**Why this exists:** ByteDance/Ouro's paper describes inference-time adaptive early-exit (stop the recurrent loop once accumulated exit probability crosses `early_exit_threshold`). Confirmed by reading vLLM's own `OuroForCausalLM` source (`vllm/model_executor/models/ouro.py`): it loads the `early_exit_gate` module's weights but never calls it -- `forward()` unconditionally runs the full `range(total_ut_steps)` loop every time, regardless of `early_exit_threshold`. Ouro's own model card confirms this same limitation. `native_infer.py` runs the model's own native code (via `trust_remote_code=True`) instead, so this notebook can show whether early exit actually behaves differently there.

**Not runnable in this sandbox** (no GPU, no model access) -- written to be copied into an environment that has both. Keep `LIMIT` small: `native_infer.py` has no batching/PagedAttention, so it's meaningfully slower per token than vLLM.

In [ ]:
import sys
sys.path.append("/home/l069561/project/meddata/")
sys.path.append("/home/l069561/project/meddata/inference_and_eval/inference/")

import json
import time
from pathlib import Path

from inference_and_eval.inference import native_infer
from inference_and_eval.inference import vllm_structured_infer
from inference_and_eval.eval.run_eval import run_eval

OUTPUT_DIR = Path("/home/l069561/project/outputs/ouro_native_vs_vllm")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Config

`DATASET` defaults to a small-output task (`2018_n2c2_track1`, gold max ~7 tokens) so a first run is fast; swap to something with a longer/JSON-shaped target (e.g. `2018_n2c2_track2`) once you've confirmed both paths load, since that's where a real difference in output length/shape between "always full depth" (vLLM) and genuine early-exit (native) would actually show up. Keep `LIMIT` small -- `native_infer.py` runs one example at a time, no batching.

In [ ]:
DATASET = "2018_n2c2_track1"  # multitask/data/eval/<DATASET>.jsonl

# Local Ouro checkpoint (matches run_custom_model_eval.ipynb's convention) --
# or point both at the same stock "ByteDance/Ouro-1.4B"/"ByteDance/Ouro-2.6B"
# Hub id if you want to compare on the base (non-fine-tuned) model instead.
MODEL = "/home/l069561/project/outputs/loop_transformer/models/ouro/1.4b/checkpoint-25860"
BACKEND = "xgrammar"          # vLLM structured-output backend
TRUST_REMOTE_CODE = True      # required for Ouro's custom modeling code, both paths

LIMIT = 20
MAX_MODEL_LEN = 32768         # vLLM only -- see run_custom_model_eval.ipynb's own note
                              # on prompt_tokens + max_tokens <= max_model_len

print(f"DATASET={DATASET!r}  MODEL={MODEL!r}  LIMIT={LIMIT}")

## Run vLLM

`_make_llm` in `vllm_structured_infer.py` already unconditionally passes `trust_remote_code=True` to `vllm.LLM(...)` (needed for Ouro regardless), so `TRUST_REMOTE_CODE` above isn't threaded in here -- it's only used by the native path below. `temperature=0.0` (greedy) on both runs for a fair, deterministic comparison.

In [ ]:
t0 = time.perf_counter()
llm = vllm_structured_infer._make_llm(MODEL, BACKEND, llm_kwargs={"max_model_len": MAX_MODEL_LEN})
completions_vllm = vllm_structured_infer.run(
    DATASET, MODEL, BACKEND, limit=LIMIT, llm=llm,
    out_path=OUTPUT_DIR / "completions_vllm.json",
)
vllm_seconds = time.perf_counter() - t0
print(f"\nvLLM: {vllm_seconds:.1f}s for {len(completions_vllm)} examples "
      f"({vllm_seconds / max(len(completions_vllm), 1):.2f}s/example)")

## Run native `transformers`

Loads its own separate model instance (not reusing `llm` above) since vLLM and plain `transformers` can't share one loaded checkpoint. If you're on a single GPU and this OOMs, free the vLLM instance first (`del llm`) before running this cell.

In [ ]:
t0 = time.perf_counter()
completions_native = native_infer.run(
    DATASET, MODEL, limit=LIMIT, trust_remote_code=TRUST_REMOTE_CODE,
    out_path=OUTPUT_DIR / "completions_native.json",
)
native_seconds = time.perf_counter() - t0
print(f"\nnative: {native_seconds:.1f}s for {len(completions_native)} examples "
      f"({native_seconds / max(len(completions_native), 1):.2f}s/example)")
print(f"native took {native_seconds / max(vllm_seconds, 1e-9):.1f}x as long as vLLM for the same {LIMIT} examples")

## Compare completions

Exact string match isn't the interesting signal here (free-form native decoding vs. grammar-constrained vLLM decoding will differ in formatting even when both are "correct" -- see `native_infer.py`'s docstring). What matters more for the early-exit question specifically: per-example latency variance under native (if early exit is really triggering per-example, some examples should take noticeably less wall-clock/fewer forward passes than others -- vLLM's is always uniform, full-depth, so its own per-example latency should be flat by comparison) and the raw completion text/length, to sanity-check native's output isn't degenerate before trusting any latency reading.

In [ ]:
shared_ids = sorted(set(completions_vllm) & set(completions_native))
print(f"{len(shared_ids)} ids in both runs\n")

for doc_id in shared_ids[:5]:
    v = completions_vllm[doc_id]
    n = completions_native[doc_id]
    v_text = v if isinstance(v, str) else json.dumps(v)
    n_text = n if isinstance(n, str) else json.dumps(n)
    match = "MATCH" if v_text.strip() == n_text.strip() else "differ"
    print(f"--- {doc_id} ({match}) ---")
    print(f"  vllm   [{len(v_text):4d} chars]: {v_text[:200]!r}")
    print(f"  native [{len(n_text):4d} chars]: {n_text[:200]!r}")
    print()

exact_matches = sum(
    1 for doc_id in shared_ids
    if (completions_vllm[doc_id] if isinstance(completions_vllm[doc_id], str) else json.dumps(completions_vllm[doc_id])).strip()
    == (completions_native[doc_id] if isinstance(completions_native[doc_id], str) else json.dumps(completions_native[doc_id])).strip()
)
print(f"exact text match: {exact_matches}/{len(shared_ids)}")

## Score both

`run_eval` scores whatever's in each completions dict the same way regardless of which script produced it. Expect `native`'s score to look worse on JSON-shaped IE tasks specifically because it has no grammar constraint (see `native_infer.py`'s docstring) -- that's a decoding-method difference, not evidence about early-exit correctness. `2018_n2c2_track1`'s task (`cohort_eligibility`) is a tiny classification-shaped target, a reasonable first sanity check before trying a bigger dataset.

In [ ]:
print("=== vLLM ===")
run_eval(DATASET, completions_vllm)

print("\n=== native ===")
run_eval(DATASET, completions_native)

## Known caveats

- **This doesn't conclusively prove early exit is or isn't happening** -- it's a starting signal, not a verdict. The `seconds/token` spread printed by the native run cell is the main thing to look at: genuinely flat (all examples ~equal seconds/token) suggests every example still ran the full `total_ut_steps` loop even natively; real spread suggests some examples are exiting earlier than others. Wall-clock on a GPU is noisy (thermal throttling, other jobs, warm-up on the first call) -- don't trust a single small `LIMIT` run's numbers, and consider excluding the very first example (cold-start) from any summary stat.
- **Native's lack of grammar-constrained decoding will lower its score on JSON-shaped IE tasks regardless of early-exit correctness** -- a bad-JSON completion there is a decoding-method artifact, not evidence the model itself is wrong. If you want an apples-to-apples *quality* comparison (not just an early-exit behavior check), that would need constrained decoding added to `native_infer.py` (e.g. via `outlines`' `transformers` integration) -- not done here; this tool was built specifically to probe early-exit, not to replace vLLM's constrained-decoding numbers.
- **`native_infer.py` has no retry/fallback/repetition-detection** (unlike `vllm_structured_infer.run()`) -- a single bad native completion is stored as-is, not re-drawn. Don't read a single native failure as proof of anything without checking the raw text.
- **Two full model loads if both cells run in the same process** -- if you're tight on GPU memory, `del llm` (and `torch.cuda.empty_cache()`) before the native run cell, or run the two halves of this notebook as two separate processes/kernels.
- Nothing in this notebook has been executed in this sandbox (no GPU, no model access) -- copy it into an environment that has both before trusting any of the printed numbers.